# Prediction Pipeline

This pipeline should run every time I want match predictions. For now the pipeline should do the following:
1. Accept the home squad id and away squad id of the teams playing
2. Get players in squads and their statistics
3. Arrange players into teams
4. Output features
5. Run model on features
6. Display prediction

In [2]:
# Imports and constants
import numpy as np
import numpy.typing as npt
import pandas as pd
from helpers.prediction import uncondense_parameters, f_x, interpret_probabilities, get_team_statistics_from_squad, get_feature_to_predict

player_statistics_file = "data/player_statistics_2026-06-15 22:25:30.781664.csv"
squad_members = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Squad Members")
statistics = pd.read_csv(player_statistics_file)
players = pd.read_excel('data/World Cup 2026.xlsx', sheet_name="Players")

model_weights_concatenated = np.load("parameters/model_V3_parameters_2026-06-12 16:16:55.752520.npy", allow_pickle=True)
mean_std = np.load("parameters/model_v3_mean_std_2026-06-12 16:17:53.057086.npy")

/home/roman/Code/AIEngineering/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
squad_players = get_team_statistics_from_squad(
    statistics=statistics, 
    players=players, 
    squad_members=squad_members, 
    squad_id=1,
    match_date="2026-06-11"
)
print(squad_players)

[[ 26.           0.           0.           0.           1.
    0.         144.           0.           3.        ]
 [ 26.           0.           1.           0.           0.
    1.         197.           9.           1.75      ]
 [ 29.           0.           0.           1.           0.
    2.         300.          16.           3.25      ]
 [ 27.           0.           0.           1.           0.
    1.         249.          10.           2.125     ]
 [ 31.           0.           0.           1.           0.
    2.         336.          41.           4.        ]
 [ 26.           0.           1.           0.           0.
    1.         224.           1.           4.        ]
 [ 29.           0.           1.           0.           0.
    0.         337.          35.           4.        ]
 [ 25.           1.           0.           0.           0.
    1.         191.          70.           3.        ]
 [ 27.           0.           1.           0.           0.
    2.         347.          

In [4]:
get_feature_to_predict(
    mean_std=mean_std,
    statistics=statistics,
    players=players,
    squad_members=squad_members,
    home_squad_id=1, 
    away_squad_id=2, 
    match_date="2026-06-11"
)

array([[-0.1162319 ],
       [-0.99144741],
       [-0.98859654],
       [-0.99144741],
       [-0.99714914],
       [-0.96293876],
       [ 7.64952388],
       [ 0.40832722],
       [-0.8991846 ],
       [-0.13618796],
       [-0.99429827],
       [-0.99144741],
       [-0.98574568],
       [-0.99714914],
       [-0.96293876],
       [ 4.06028506],
       [-0.44693222],
       [-0.89416956]])

In [8]:
def make_game_prediction(
    mean_std: npt.NDArray,
    statistics: pd.DataFrame,
    players: pd.DataFrame,
    squad_members: pd.DataFrame,
    home_squad: int, 
    away_squad: int,
    match_date: int
):
    """  
    Make prediction of outcome between home squad and away squad using weights of learnt model
    
    Args:
        home_squad (scalar): ID of home squad
        away_squad (scalar): ID of away squad
        match_date (scalar): Date match is being played in YYYY-MM-dd
    """
    feature = get_feature_to_predict(
        mean_std=mean_std,
        players=players,
        statistics=statistics,
        squad_members=squad_members,
        home_squad_id=home_squad, 
        away_squad_id=away_squad, 
        match_date=match_date
    )
    W1, b1, W2, b2, W3, b3 = uncondense_parameters(model_weights_concatenated)
    pred, _, _, _, _ = f_x(X=feature, W1=W1, W2=W2, W3=W3, b1=b1, b2=b2, b3=b3)
    prediction = interpret_probabilities(pred)
    print(prediction)
    
make_game_prediction(
    mean_std=mean_std,
    players=players,
    statistics=statistics,
    squad_members=squad_members,    
    home_squad=24,
    away_squad=23, 
    match_date="2026-06-11"
)
    

0-3
